# 02 – Thiết kế thực nghiệm và baseline mô hình dự đoán số sao / cảm xúc
Buổi 5 – Thiết kế chi tiết. Người thực hiện chính: KAYSONE Souk Anong.
Thiết kế: tách 70/10/20 (train/validation/test) phân tầng theo số sao sau khi bỏ trùng;
chọn siêu tham số trên validation; huấn luyện lại trên 80% (train+validation) và đánh giá MỘT lần trên test 20%.

In [1]:
import sys, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import matplotlib as mpl
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix
from backend.core import config as C
from backend.services import data_service as DS
import vizstyle as V
V.apply()
FIG = ROOT / "notebooks" / "figures"; OUT = ROOT / "notebooks" / "outputs"
R = {}

In [2]:
df = pd.read_csv(C.DATA_PROCESSED / "reviews_clean.csv", usecols=["review_id", "score", "year_month", "summary", "text"],
                 dtype={"summary": str, "text": str}, keep_default_na=False)
df["doc"] = df.summary + " . " + df.text          # FR-10: dùng cả tiêu đề và nội dung
y = df.score.values
idx = np.arange(len(df))
i_trval, i_test = train_test_split(idx, test_size=C.TEST_SIZE, stratify=y, random_state=C.RANDOM_STATE)
i_train, i_val = train_test_split(i_trval, test_size=C.VAL_SIZE_OF_TRAIN, stratify=y[i_trval], random_state=C.RANDOM_STATE)
R["split"] = {"n_total": len(df), "train": len(i_train), "val": len(i_val), "test": len(i_test),
              "test_score_dist": {int(k): int(v) for k, v in pd.Series(y[i_test]).value_counts().sort_index().items()}}
R["split"]

{'n_total': 393890,
 'train': 275723,
 'val': 39389,
 'test': 78778,
 'test_score_dist': {1: 7260, 2: 4160, 3: 5954, 4: 11217, 5: 50187}}

## Hàm đánh giá dùng chung (thước đo theo NFR-03)

In [3]:
SENT = np.array([0, 0, 0, 1, 2, 2])          # chỉ số theo số sao: 1–2 → 0 (Tiêu cực), 3 → 1, 4–5 → 2
def sent_from_proba(P):                     # P: n × 5 (1..5 sao) → cảm xúc theo tổng xác suất
    return np.argmax(np.c_[P[:, 0] + P[:, 1], P[:, 2], P[:, 3] + P[:, 4]], axis=1)

def evaluate(y_true, P=None, y_pred=None):
    if P is not None: y_pred = P.argmax(1) + 1
    s_true = SENT[y_true]; s_pred = sent_from_proba(P) if P is not None else SENT[y_pred]
    p, r, f, _ = precision_recall_fscore_support(s_true, s_pred, labels=[0, 1, 2], zero_division=0)
    out = {"f1_macro3": f1_score(s_true, s_pred, average="macro"), "f1_neg": f[0], "f1_neu": f[1], "f1_pos": f[2],
           "prec_neg": p[0], "rec_neg": r[0], "prec_neu": p[1], "rec_neu": r[1], "prec_pos": p[2], "rec_pos": r[2],
           "acc3": accuracy_score(s_true, s_pred), "acc5": accuracy_score(y_true, y_pred),
           "f1_macro5": f1_score(y_true, y_pred, average="macro"), "mae": float(np.abs(y_true - y_pred).mean())}
    if P is not None: out["mae_expected"] = float(np.abs(y_true - (P * np.arange(1, 6)).sum(1)).mean())
    return {k: round(float(v), 4) for k, v in out.items()}

def make_vec(): return TfidfVectorizer(preprocessor=DS.model_preprocess, ngram_range=C.TFIDF_NGRAM, min_df=C.TFIDF_MIN_DF,
                                       max_features=C.TFIDF_MAX_FEATURES, sublinear_tf=True, dtype=np.float32)

## B0 – Baseline đơn giản nhất: luôn dự đoán lớp đông nhất (5 sao)

In [4]:
R["B0_majority"] = evaluate(y[i_test], y_pred=np.full(len(i_test), 5)); R["B0_majority"]

{'f1_macro3': 0.292,
 'f1_neg': 0.0,
 'f1_neu': 0.0,
 'f1_pos': 0.8761,
 'prec_neg': 0.0,
 'rec_neg': 0.0,
 'prec_neu': 0.0,
 'rec_neu': 0.0,
 'prec_pos': 0.7795,
 'rec_pos': 1.0,
 'acc3': 0.7795,
 'acc5': 0.6371,
 'f1_macro5': 0.1557,
 'mae': 0.8206}

## Chọn siêu tham số trên tập validation

In [5]:
# Chọn siêu tham số trên mẫu phân tầng 100.000 dòng của tập train (đúng quy mô huấn luyện của ứng dụng),
# đánh giá trên toàn bộ tập validation.
i_tune = train_test_split(i_train, train_size=C.TRAIN_SAMPLE_MAX, stratify=y[i_train], random_state=C.RANDOM_STATE)[0]
t0 = time.time(); vec = make_vec(); Xtr = vec.fit_transform(df.doc.values[i_tune]); Xva = vec.transform(df.doc.values[i_val])
R["tfidf_fit_seconds_100k"] = round(time.time() - t0, 1); R["vocab_size_100k"] = len(vec.vocabulary_)
ytr, yva = y[i_tune], y[i_val]; sw = compute_sample_weight("balanced", ytr)
grid = {"LR": [0.5, 1.0, 2.0, 4.0], "NB": [0.03, 0.1, 0.3, 1.0], "SVM": [0.05, 0.1, 0.3]}
tune = []
for c in grid["LR"]:
    t0 = time.time(); m = LogisticRegression(C=c, class_weight="balanced", max_iter=300, solver="lbfgs").fit(Xtr, ytr)
    tune.append({"model": "LR", "param": f"C={c}", "sec": round(time.time() - t0, 1), **evaluate(yva, P=m.predict_proba(Xva))}); print(tune[-1]["model"], tune[-1]["param"], tune[-1]["f1_macro3"], flush=True)
for a in grid["NB"]:
    t0 = time.time(); m = MultinomialNB(alpha=a).fit(Xtr, ytr, sample_weight=sw)
    tune.append({"model": "NB", "param": f"alpha={a}", "sec": round(time.time() - t0, 1), **evaluate(yva, P=m.predict_proba(Xva))})
for c in grid["SVM"]:
    t0 = time.time(); m = CalibratedClassifierCV(LinearSVC(C=c, class_weight="balanced"), cv=3, method="sigmoid").fit(Xtr, ytr)
    tune.append({"model": "SVM", "param": f"C={c}", "sec": round(time.time() - t0, 1), **evaluate(yva, P=m.predict_proba(Xva))})
tune = pd.DataFrame(tune); tune.to_csv(OUT / "model_tuning_val.csv", index=False)
best = tune.loc[tune.groupby("model").f1_macro3.idxmax()].set_index("model")
R["tuning_best"] = best[["param", "f1_macro3", "f1_neg", "mae"]].to_dict("index")
tune[["model", "param", "sec", "f1_macro3", "f1_neg", "f1_neu", "mae", "acc5"]]

LR C=0.5 0.6925


LR C=1.0 0.7042


LR C=2.0 0.706


LR C=4.0 0.7068


,model,param,sec,f1_macro3,f1_neg,f1_neu,mae,acc5
0,LR,C=0.5,37.9,0.6925,0.7474,0.3882,0.4341,0.6918
1,LR,C=1.0,41.8,0.7042,0.7611,0.4059,0.4150,0.7000
2,LR,C=2.0,67.8,0.7060,0.7649,0.4066,0.4006,0.7073
3,LR,C=4.0,69.1,0.7068,0.7684,0.4047,0.3914,0.7144
4,NB,alpha=0.03,0.1,0.6585,0.7196,0.3204,0.4497,0.6904
5,NB,alpha=0.1,0.1,0.6749,0.7316,0.3555,0.4631,0.6796
6,NB,alpha=0.3,0.1,0.6780,0.7261,0.3724,0.4936,0.6600
7,NB,alpha=1.0,0.1,0.6695,0.7073,0.3714,0.5429,0.6255
8,SVM,C=0.05,10.7,0.6656,0.7370,0.3217,0.4035,0.7477
9,SVM,C=0.1,10.0,0.6748,0.7500,0.3335,0.3873,0.7525


## Đường học (learning curve) – căn cứ cho quyết định lấy mẫu 100.000 dòng khi file lớn

In [6]:
lc = []
bestC = float(best.loc["LR", "param"].split("=")[1])
for n in [25_000, 50_000, 100_000, 200_000]:
    sub = i_train if n == len(i_train) else train_test_split(i_train, train_size=n, stratify=y[i_train], random_state=C.RANDOM_STATE)[0]
    t0 = time.time(); v = make_vec(); Xs = v.fit_transform(df.doc.values[sub])
    m = LogisticRegression(C=bestC, class_weight="balanced", max_iter=300).fit(Xs, y[sub]); sec = time.time() - t0
    lc.append({"n_train": n, "sec_fit_total": round(sec, 1), **evaluate(yva, P=m.predict_proba(v.transform(df.doc.values[i_val])))}); print("LC", n, lc[-1]["f1_macro3"], flush=True)
lc = pd.DataFrame(lc); lc.to_csv(OUT / "learning_curve.csv", index=False); R["learning_curve"] = lc.to_dict("records")
lc[["n_train", "sec_fit_total", "f1_macro3", "f1_neg", "mae"]]

LC 25000 0.6838


LC 50000 0.6994


LC 100000 0.7068


/home/claude/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LC 200000 0.7189


,n_train,sec_fit_total,f1_macro3,f1_neg,mae
0,25000,17.0,0.6838,0.7447,0.4158
1,50000,45.9,0.6994,0.7583,0.4005
2,100000,78.7,0.7068,0.7684,0.3914
3,200000,161.5,0.7189,0.7772,0.3862


## Huấn luyện lại cấu hình tốt nhất trên 80% và đánh giá một lần trên test 20%

In [7]:
t0 = time.time(); vecF = make_vec(); Xtv = vecF.fit_transform(df.doc.values[i_trval]); Xte = vecF.transform(df.doc.values[i_test])
ytv, yte = y[i_trval], y[i_test]; swF = compute_sample_weight("balanced", ytv)
R["tfidf_fit_seconds_80"] = round(time.time() - t0, 1)
final = {}
for name in ["LR", "NB", "SVM"]:
    prm = float(best.loc[name, "param"].split("=")[1]); t0 = time.time()
    if name == "LR": m = LogisticRegression(C=prm, class_weight="balanced", max_iter=300).fit(Xtv, ytv)
    elif name == "NB": m = MultinomialNB(alpha=prm).fit(Xtv, ytv, sample_weight=swF)
    else: m = CalibratedClassifierCV(LinearSVC(C=prm, class_weight="balanced"), cv=3, method="sigmoid").fit(Xtv, ytv)
    sec = round(time.time() - t0, 1); P = m.predict_proba(Xte)
    final[name] = {"model": m, "P": P, "res": {"param": best.loc[name, "param"], "fit_seconds": sec, **evaluate(yte, P=P)}}; print("FINAL", name, final[name]["res"]["f1_macro3"], flush=True)
R["test"] = {k: v["res"] for k, v in final.items()}
pd.DataFrame(R["test"]).T

FINAL LR 0.7193


/home/claude/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


FINAL NB 0.6846


FINAL SVM 0.6988


,param,fit_seconds,f1_macro3,f1_neg,f1_neu,f1_pos,prec_neg,rec_neg,prec_neu,rec_neu,prec_pos,rec_pos,acc3,acc5,f1_macro5,mae,mae_expected
LR,C=4.0,181.9,0.7193,0.7812,0.4274,0.9494,0.7407,0.8264,0.4435,0.4125,0.9565,0.9424,0.8855,0.7093,0.5511,0.3849,0.4871
NB,alpha=0.3,0.4,0.6846,0.7315,0.384,0.9382,0.6425,0.8492,0.4581,0.3305,0.9542,0.9228,0.8674,0.671,0.5198,0.472,0.7371
SVM,C=0.3,218.6,0.6988,0.7842,0.364,0.9482,0.8366,0.738,0.5826,0.2647,0.9152,0.9837,0.8937,0.7614,0.5337,0.3524,0.4688


## Chế độ của ứng dụng: huấn luyện trên mẫu phân tầng 100.000 dòng, đánh giá trên cùng tập test

In [8]:
sub = train_test_split(i_trval, train_size=C.TRAIN_SAMPLE_MAX, stratify=y[i_trval], random_state=C.RANDOM_STATE)[0]
t0 = time.time(); v100 = make_vec(); X100 = v100.fit_transform(df.doc.values[sub])
m100 = LogisticRegression(C=bestC, class_weight="balanced", max_iter=300).fit(X100, y[sub]); sec = time.time() - t0
R["app_mode_100k"] = {"fit_seconds_total": round(sec, 1), **evaluate(yte, P=m100.predict_proba(v100.transform(df.doc.values[i_test])))}
R["app_mode_100k"]

{'fit_seconds_total': 79.6,
 'f1_macro3': 0.7071,
 'f1_neg': 0.769,
 'f1_neu': 0.4045,
 'f1_pos': 0.9479,
 'prec_neg': 0.7303,
 'rec_neg': 0.8119,
 'prec_neu': 0.4523,
 'rec_neu': 0.3658,
 'prec_pos': 0.9489,
 'rec_pos': 0.9468,
 'acc3': 0.8833,
 'acc5': 0.7138,
 'f1_macro5': 0.5418,
 'mae': 0.3907,
 'mae_expected': 0.5253}

## Đối chứng thiết kế QĐ-03: mô hình 3 lớp trực tiếp so với mô hình 5 sao suy ra 3 lớp (cùng mẫu 100.000 dòng)

In [9]:
m3 = LogisticRegression(C=bestC, class_weight="balanced", max_iter=300).fit(X100, SENT[y[sub]])
s_pred = m3.predict(v100.transform(df.doc.values[i_test]))
R["ablation_direct3_100k"] = {"f1_macro3": round(f1_score(SENT[yte], s_pred, average="macro"), 4),
                              "f1_neg": round(f1_score(SENT[yte], s_pred, labels=[0], average="macro"), 4),
                              "compare_5to3_100k": {k: R["app_mode_100k"][k] for k in ["f1_macro3", "f1_neg"]}}
R["ablation_direct3_100k"]

{'f1_macro3': 0.7157,
 'f1_neg': 0.7673,
 'compare_5to3_100k': {'f1_macro3': 0.7071, 'f1_neg': 0.769}}

## Kiểm tra trôi dữ liệu theo thời gian: huấn luyện trước 2012, kiểm tra trên dữ liệu năm 2012

In [10]:
old = np.where(df.year_month.values < "2012-01")[0]; new = np.where(df.year_month.values >= "2012-01")[0]
sub_old = train_test_split(old, train_size=C.TRAIN_SAMPLE_MAX, stratify=y[old], random_state=C.RANDOM_STATE)[0]
v_o = make_vec(); m_o = LogisticRegression(C=bestC, class_weight="balanced", max_iter=300).fit(v_o.fit_transform(df.doc.values[sub_old]), y[sub_old])
R["out_of_time_2012"] = {"n_test": len(new), **evaluate(y[new], P=m_o.predict_proba(v_o.transform(df.doc.values[new])))}
R["out_of_time_2012"]

{'n_test': 139179,
 'f1_macro3': 0.7081,
 'f1_neg': 0.779,
 'f1_neu': 0.4026,
 'f1_pos': 0.9427,
 'prec_neg': 0.7453,
 'rec_neg': 0.8159,
 'prec_neu': 0.4342,
 'rec_neu': 0.3753,
 'prec_pos': 0.9455,
 'rec_pos': 0.9398,
 'acc3': 0.8754,
 'acc5': 0.7006,
 'f1_macro5': 0.541,
 'mae': 0.411,
 'mae_expected': 0.5422}

## Ma trận nhầm lẫn, độ trễ dự đoán, đánh giá lệch, từ khóa ảnh hưởng (mô hình được chọn)

In [11]:
sel = max(R["tuning_best"], key=lambda k: R["tuning_best"][k]["f1_macro3"]); R["selected"] = sel   # chọn trên validation; test chỉ để báo cáo
Psel = final[sel]["P"]; msel = final[sel]["model"]
cm3 = confusion_matrix(SENT[yte], sent_from_proba(Psel), labels=[0, 1, 2]); cm5 = confusion_matrix(yte, Psel.argmax(1) + 1, labels=[1, 2, 3, 4, 5])
R["confusion3"] = cm3.tolist(); R["confusion5"] = cm5.tolist()
# độ trễ: vector hóa + dự đoán 1 đánh giá, lặp 200 lần
docs = df.doc.values[i_test[:200]]; t0 = time.perf_counter()
for d in docs: msel.predict_proba(vecF.transform([d]))
R["latency_ms_per_review"] = round((time.perf_counter() - t0) / len(docs) * 1000, 2)
# đánh giá lệch (FR-13): 5 sao nhưng P(Tiêu cực) >= 0,8 và ngược lại
pneg = Psel[:, 0] + Psel[:, 1]; ppos = Psel[:, 3] + Psel[:, 4]
R["mismatch_test"] = {"5star_but_neg": int(((yte == 5) & (pneg >= C.MISMATCH_PROB)).sum()),
                      "1star_but_pos": int(((yte == 1) & (ppos >= C.MISMATCH_PROB)).sum())}
ex = df.iloc[i_test[(yte == 5) & (pneg >= C.MISMATCH_PROB)][:3]][["review_id", "score", "summary"]]
R["mismatch_examples"] = ex.to_dict("records")
# từ khóa ảnh hưởng: w_j = Σ_k (k − 3)·β_kj trên mô hình LR
lr = final["LR"]["model"]; w = ((np.arange(1, 6) - 3)[:, None] * lr.coef_).sum(0)
vocab = np.array(vecF.get_feature_names_out()); o = np.argsort(w)
R["keywords_up"] = dict(zip(vocab[o[::-1][:15]], np.round(w[o[::-1][:15]], 2).tolist()))
R["keywords_down"] = dict(zip(vocab[o[:15]], np.round(w[o[:15]], 2).tolist()))
R["latency_ms_per_review"], R["mismatch_test"], R["keywords_down"]

(2.47,
 {'5star_but_neg': 210, '1star_but_pos': 64},
 {'not': -71.02,
  'not good': -59.72,
  'not worth': -59.13,
  'disappointing': -57.14,
  'not recommend': -53.66,
  'yuck': -53.61,
  'terrible': -52.28,
  'awful': -49.75,
  'worst': -47.99,
  'disappointed': -47.69,
  'horrible': -46.51,
  'disappointment': -45.62,
  'very disappointed': -42.25,
  'not great': -41.5,
  'not very': -39.69})

In [12]:
fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.8), gridspec_kw={"wspace": 0.45})
names = ["B0", "NB", "SVM", "LR"]; lab = ["Luôn 5 sao", "Naive Bayes", "Linear SVM", "Logistic Reg."]
vals = [R["B0_majority"]] + [R["test"][k] for k in ["NB", "SVM", "LR"]]
comma = mpl.ticker.FuncFormatter(lambda x, _: f"{x:g}".replace(".", ","))
for ax, key, title, thr, better in [(axs[0], "f1_macro3", "a) F1-macro 3 lớp ↑", 0.65, "≥"), (axs[1], "f1_neg", "b) F1 lớp Tiêu cực ↑", 0.75, "≥"), (axs[2], "mae", "c) MAE số sao ↓", 0.8, "≤")]:
    v = [d[key] for d in vals]; colors = [V.GRAY, V.BLUE, V.BLUE, V.BLUE]
    bars = ax.barh(lab, v, color=colors, height=0.6); ax.axvline(thr, color=V.ORANGE, lw=1.2, ls="--")
    ax.set_title(title, pad=8); ax.grid(axis="y", visible=False); ax.set_ylim(-1.15, 3.5)
    ax.text(thr, -0.95, f" NFR-03 {better} {thr}".replace(".", ","), fontsize=7, color=V.ORANGE, ha="left", va="center")   # nhãn ngưỡng đặt dưới, không đè tiêu đề
    for b, x in zip(bars, v): ax.text(x, b.get_y() + b.get_height() / 2, f" {x:.3f}".replace(".", ","), va="center", fontsize=7, color=V.INK2)
    if ax is not axs[0]: ax.set_yticklabels([])
    ax.set_xlim(0, max(max(v), thr) * 1.32); ax.xaxis.set_major_formatter(comma)
V.save(fig, FIG / "model_01_compare.png")

fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0), gridspec_kw={"wspace": 0.4})
for ax, cm, labs, title in [(axs[0], cm3, ["Tiêu cực", "Trung tính", "Tích cực"], "a) 3 lớp cảm xúc"), (axs[1], cm5, ["1", "2", "3", "4", "5"], "b) 5 mức sao")]:
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cmn[i, j]:.0%}", ha="center", va="center", fontsize=7.5, color="white" if cmn[i, j] > 0.55 else V.INK)
    ax.set_xticks(range(len(labs)), labs, fontsize=7.5); ax.set_yticks(range(len(labs)), labs, fontsize=7.5)
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế"); ax.set_title(title); ax.grid(False)
V.save(fig, FIG / "model_02_confusion.png")

fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.plot(lc.n_train / 1000, lc.f1_macro3, marker="o", color=V.BLUE, label="F1-macro 3 lớp")
ax.plot(lc.n_train / 1000, lc.f1_neg, marker="o", color=V.RED, label="F1 lớp Tiêu cực")
ax.axvline(100, color=V.ORANGE, ls="--", lw=1.2); ax.text(102, lc.f1_macro3.min(), "mẫu 100.000 dòng\n(chế độ ứng dụng)", fontsize=7, color=V.ORANGE)
ax.set_xlabel("Số đánh giá dùng để huấn luyện (nghìn)"); ax.set_title("Đường học của TF-IDF + Logistic Regression (đo trên validation)"); ax.legend(loc="lower right")
V.save(fig, FIG / "model_03_learning_curve.png")

In [13]:
import joblib
joblib.dump(vecF, C.MODELS_DIR / "vectorizer.joblib", compress=3); joblib.dump(final[sel]["model"], C.MODELS_DIR / "star_model.joblib", compress=3)
(OUT / "model_results.json").write_text(json.dumps(R, ensure_ascii=False, indent=1, default=float), encoding="utf-8")
print(json.dumps({k: R[k] for k in ["test", "selected", "app_mode_100k", "out_of_time_2012", "ablation_direct3_100k", "latency_ms_per_review"]}, ensure_ascii=False, indent=1, default=float))

{
 "test": {
  "LR": {
   "param": "C=4.0",
   "fit_seconds": 181.9,
   "f1_macro3": 0.7193,
   "f1_neg": 0.7812,
   "f1_neu": 0.4274,
   "f1_pos": 0.9494,
   "prec_neg": 0.7407,
   "rec_neg": 0.8264,
   "prec_neu": 0.4435,
   "rec_neu": 0.4125,
   "prec_pos": 0.9565,
   "rec_pos": 0.9424,
   "acc3": 0.8855,
   "acc5": 0.7093,
   "f1_macro5": 0.5511,
   "mae": 0.3849,
   "mae_expected": 0.4871
  },
  "NB": {
   "param": "alpha=0.3",
   "fit_seconds": 0.4,
   "f1_macro3": 0.6846,
   "f1_neg": 0.7315,
   "f1_neu": 0.384,
   "f1_pos": 0.9382,
   "prec_neg": 0.6425,
   "rec_neg": 0.8492,
   "prec_neu": 0.4581,
   "rec_neu": 0.3305,
   "prec_pos": 0.9542,
   "rec_pos": 0.9228,
   "acc3": 0.8674,
   "acc5": 0.671,
   "f1_macro5": 0.5198,
   "mae": 0.472,
   "mae_expected": 0.7371
  },
  "SVM": {
   "param": "C=0.3",
   "fit_seconds": 218.6,
   "f1_macro3": 0.6988,
   "f1_neg": 0.7842,
   "f1_neu": 0.364,
   "f1_pos": 0.9482,
   "prec_neg": 0.8366,
   "rec_neg": 0.738,
   "prec_neu": 0.5826,
